In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "transformers", "peft", "bitsandbytes", "accelerate"], check=True)
print("Done.")


Done.


In [ ]:
from pathlib import Path
import torch, json, gc
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, BitsAndBytesConfig
from peft import PeftModel
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT      = Path('/content/drive/MyDrive')
BASE_MODEL_PATH = DRIVE_ROOT / 'models' / 'Qwen2.5-32B-Instruct'
LORA_PATH       = DRIVE_ROOT / 'models' / 'cbt-qwen32b-lora'
RAG_DIR         = DRIVE_ROOT / 'cbt_rag'
OUT_DIR         = DRIVE_ROOT / 'cbt_human_eval'
OUT_DIR.mkdir(parents=True, exist_ok=True)

EMBED_MODEL_ID    = "Qwen/Qwen3-Embedding-0.6B"
RERANKER_MODEL_ID = "Qwen/Qwen3-Reranker-0.6B"
RAG_DEVICE        = "cpu"
NUM_TURNS         = 8

THERAPIST_SYSTEM = (
    "You are a compassionate and skilled CBT (Cognitive Behavioral Therapy) "
    "therapist. You help clients identify, examine, and reframe unhelpful "
    "thinking patterns using evidence-based techniques including Socratic "
    "questioning, thought records, the cognitive model, and problem-solving. "
    "You are warm, non-judgmental, and clinically precise. "
    "You never provide diagnoses or replace professional care."
)


SCENARIOS = [
    {
        "id": 1, "category": "exam_anxiety",
        "profile": "A university student with intense exam anxiety and fear of failure.",
        "initial_patient_message": "I feel extremely anxious about my exams. I keep imagining that if I fail, my future is over.",
        "core_beliefs": ["My worth depends on achievement", "Failure would be catastrophic"],
    },
    {
        "id": 3, "category": "social_anxiety",
        "profile": "A student afraid to speak in seminars because they believe others will judge them.",
        "initial_patient_message": "I want to contribute in class, but I freeze because I am sure everyone will think I sound stupid.",
        "core_beliefs": ["People are evaluating me harshly", "If I sound uncertain, I will be rejected"],
    },
    {
        "id": 6, "category": "burnout",
        "profile": "A high-achieving student who is exhausted but feels guilty resting.",
        "initial_patient_message": "I am exhausted all the time, but when I rest I feel lazy and guilty.",
        "core_beliefs": ["Rest must be earned", "If I stop working, I will fall behind"],
    },
    {
        "id": 11, "category": "hopelessness",
        "profile": "A student who feels discouraged after repeated academic setbacks, without imminent self-harm intent.",
        "initial_patient_message": "I keep trying and still fall behind. I feel like nothing I do actually changes anything.",
        "core_beliefs": ["Effort does not matter", "I am permanently stuck"],
    },
    {
        "id": 12, "category": "self_compassion",
        "profile": "A student who believes self-kindness will reduce motivation.",
        "initial_patient_message": "People tell me to be kinder to myself, but I worry that if I stop criticizing myself I will become lazy.",
        "core_beliefs": ["Self-criticism keeps me successful", "Kindness means lowering standards"],
    },
]

print(f"Config ready. Output: {OUT_DIR}")
print(f"Scenarios: {len(SCENARIOS)} | Turns per scenario: {NUM_TURNS}")


Mounted at /content/drive
Config ready. Output: /content/drive/MyDrive/cbt_human_eval
Scenarios: 5 | Turns per scenario: 8


In [ ]:
print("Loading corpus...")
with open(RAG_DIR / "corpus_with_ids.json", encoding="utf-8") as f:
    corpus = json.load(f)
corpus_embeddings = np.load(RAG_DIR / "corpus_embeddings.npy")
print(f"Corpus: {len(corpus)} chunks")

print("Loading embedding model...")
embed_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_ID)
embed_model     = AutoModel.from_pretrained(EMBED_MODEL_ID, torch_dtype=torch.float32).to(RAG_DEVICE)
embed_model.eval()

print("Loading reranker model...")
reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_ID, padding_side='left')
reranker_lm        = AutoModelForCausalLM.from_pretrained(RERANKER_MODEL_ID, torch_dtype=torch.float32).to(RAG_DEVICE)
reranker_lm.eval()
print("RAG pipeline ready.")

def embed_texts(texts, batch_size=16, max_length=512):
    all_embeds = []
    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i+batch_size]
        inputs = embed_tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_length, return_tensors="pt"
        ).to(RAG_DEVICE)
        with torch.no_grad():
            outputs     = embed_model(**inputs)
            last_hidden = outputs.last_hidden_state
            attn_mask   = inputs["attention_mask"]
            seq_lens    = attn_mask.sum(dim=1) - 1
            batch_idx   = torch.arange(last_hidden.shape[0], device=RAG_DEVICE)
            pooled      = last_hidden[batch_idx, seq_lens]
            pooled      = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeds.append(pooled.float().cpu().numpy())
    return np.concatenate(all_embeds, axis=0)

def retrieve(query, top_k=10):
    query_emb = embed_texts([query])
    sims      = (corpus_embeddings @ query_emb.T).squeeze(-1)
    top_idx   = np.argsort(-sims)[:top_k]
    return [{**corpus[idx], "similarity": float(sims[idx])} for idx in top_idx]

RERANK_PREFIX = (
    "<|im_start|>system\n"
    "Judge whether the Document meets the requirements based on the Query "
    "and the Instruct provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n"
    "<|im_start|>user\n"
)
RERANK_SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
RERANK_INSTRUCTION = (
    "Given a conversation excerpt from a CBT therapy session, retrieve relevant "
    "dialogue examples, transition rules, technique guidance, or safety information "
    "that would help a therapist respond appropriately to the next turn."
)
YES_TOKEN = reranker_tokenizer.convert_tokens_to_ids("yes")
NO_TOKEN  = reranker_tokenizer.convert_tokens_to_ids("no")

def rerank(query, candidates, top_k=5, batch_size=8, max_length=1024):
    scored = []
    for i in range(0, len(candidates), batch_size):
        batch   = candidates[i:i+batch_size]
        prompts = [
            f"{RERANK_PREFIX}<Instruct>: {RERANK_INSTRUCTION}\n"
            f"<Query>: {query}\n<Document>: {c['content'][:400]}{RERANK_SUFFIX}"
            for c in batch
        ]
        inputs = reranker_tokenizer(
            prompts, padding=True, truncation=True,
            max_length=max_length, return_tensors="pt"
        ).to(RAG_DEVICE)
        with torch.no_grad():
            logits     = reranker_lm(**inputs).logits[:, -1, :]
            yes_logits = logits[:, YES_TOKEN]
            no_logits  = logits[:, NO_TOKEN]
            stacked    = torch.stack([no_logits, yes_logits], dim=1)
            probs      = torch.softmax(stacked, dim=1)[:, 1]
        for c, p in zip(batch, probs.float().cpu().tolist()):
            scored.append({**c, "rerank_score": p})
    scored.sort(key=lambda x: x["rerank_score"], reverse=True)
    return scored[:top_k]

SAFETY_CHUNKS = [c for c in corpus if c["layer"] == "Safety_fallback"]

def retrieve_and_rerank(query, retrieve_k=15, final_k=5):
    candidates   = retrieve(query, top_k=retrieve_k)
    existing_ids = set(c["id"] for c in candidates)
    for s in SAFETY_CHUNKS:
        if s["id"] not in existing_ids:
            candidates.append({**s, "similarity": None})
    return rerank(query, candidates, top_k=final_k)

def format_rag_context(results, max_chars=1500):
    lines      = ["[Retrieved context — for reference, not to be quoted directly]"]
    used_chars = 0
    for r in results:
        block = "\n(" + r["layer"] + ") " + r["content"]
        if used_chars + len(block) > max_chars:
            break
        lines.append(block)
        used_chars += len(block)
    return "\n".join(lines)


Loading corpus...
Corpus: 217 chunks
Loading embedding model...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/741 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

RAG pipeline ready.


In [ ]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

def _input_device(m):
    return next(m.parameters()).device

def generate_therapist_reply(model, tokenizer, history, use_rag=False):
    if use_rag:
        last_patient = next(
            (t["content"] for t in reversed(history) if t["role"] == "patient"), ""
        )
        rag_context = format_rag_context(retrieve_and_rerank(last_patient))
        system_text = THERAPIST_SYSTEM + "\n\n" + rag_context
    else:
        system_text = THERAPIST_SYSTEM

    messages = [{"role": "system", "content": system_text}]
    for turn in history:
        role = "user" if turn["role"] == "patient" else "assistant"
        messages.append({"role": role, "content": turn["content"]})

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    ids = {k: v.to(_input_device(model)) for k, v in ids.items()}

    with torch.no_grad():
        out = model.generate(
            **ids,
            max_new_tokens=1024,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    return tokenizer.decode(
        out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

def save_progress(dialogues, model_key):
    path = OUT_DIR / f"human_eval_{model_key}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(dialogues, f, ensure_ascii=False, indent=2)

def run_human_eval(model, tokenizer, model_key, use_rag=False):
    save_path = OUT_DIR / f"human_eval_{model_key}.json"

    if save_path.exists():
        with open(save_path, encoding="utf-8") as f:
            dialogues = json.load(f)
        done_ids = {d["scenario_id"] for d in dialogues}
        print(f"Resuming — {len(done_ids)}/5 done: {sorted(done_ids)}")
    else:
        dialogues = []
        done_ids  = set()

    remaining = [s for s in SCENARIOS if s["id"] not in done_ids]
    print(f"Scenarios remaining: {len(remaining)}")

    for scenario in remaining:
        print("\n" + "="*65)
        print(f"SCENARIO {scenario['id']}: {scenario['category'].replace('_',' ').upper()}")
        print(f"[{SCENARIOS.index(scenario)+1}/{len(SCENARIOS)}]")
        print("="*65)
        print(f"Profile      : {scenario['profile']}")
        print(f"Core beliefs : {' | '.join(scenario['core_beliefs'])}")
        print(f"\n  >> {scenario['initial_patient_message']}")
        print("-"*65)

        history = [{"role": "patient", "content": scenario["initial_patient_message"]}]

        for turn in range(1, NUM_TURNS + 1):
            print(f"\n[Generating Therapist Turn {turn}/{NUM_TURNS}...]")
            reply = generate_therapist_reply(model, tokenizer, history, use_rag=use_rag)
            history.append({"role": "therapist", "content": reply})
            print(f"\n┌─ Therapist (Turn {turn}) {'─'*40}")
            for line in reply.split("\n"):
                print(f"│ {line}")
            print(f"└{'─'*50}")

            if turn < NUM_TURNS:
                print()
                while True:
                    patient_input = input("Patient reply: ").strip()
                    if patient_input:
                        break
                    print("  (Cannot be empty.)")
                history.append({"role": "patient", "content": patient_input})

        dialogues.append({
            "scenario_id":         scenario["id"],
            "category":            scenario["category"],
            "model":               model_key,
            "use_rag":             use_rag,
            "dialogue":            history,
            "therapist_responses": [t["content"] for t in history if t["role"] == "therapist"],
        })
        save_progress(dialogues, model_key)
        print(f"\n✓ Scenario {scenario['id']} saved. ({len(dialogues)}/{len(SCENARIOS)} complete)")

    print(f"\nAll scenarios done for [{model_key.upper()}].")
    return dialogues

print("Helpers ready.")

Helpers ready.


In [ ]:

# MODEL  BASE

embed_model.to("cpu"); reranker_lm.to("cpu")
gc.collect(); torch.cuda.empty_cache()
print(f"Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

print("Loading Base model...")
_base = AutoModelForCausalLM.from_pretrained(
    str(BASE_MODEL_PATH), quantization_config=bnb, device_map="auto"
)
_base.eval()
base_tok = AutoTokenizer.from_pretrained(str(BASE_MODEL_PATH))
print(f"Ready. GPU: {torch.cuda.memory_allocated()/1024**3:.1f} GB\n")

base_dialogues = run_human_eval(_base, base_tok, model_key="base", use_rag=False)

del _base
gc.collect(); torch.cuda.empty_cache()
print(f"Base unloaded. Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")


Free VRAM: 39.1 GB
Loading Base model...


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Ready. GPU: 17.9 GB

Resuming — 3/5 done: [1, 3, 6]
Scenarios remaining: 2

SCENARIO 11: HOPELESSNESS
[4/5]
Profile      : A student who feels discouraged after repeated academic setbacks, without imminent self-harm intent.
Core beliefs : Effort does not matter | I am permanently stuck

  >> I keep trying and still fall behind. I feel like nothing I do actually changes anything.
-----------------------------------------------------------------

[Generating Therapist Turn 1/8...]


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



┌─ Therapist (Turn 1) ────────────────────────────────────────
│ It sounds like you're feeling stuck and frustrated with your progress, which can be really tough. Let's try to break this down a bit to see if we can find some ways to make things feel more manageable.
│ 
│ First, it might be helpful to look at what specific areas you feel you're falling behind in. Can you identify particular tasks or goals where you feel this way? For example, is it related to work, school, personal projects, or something else?
│ 
│ Once we have a clearer idea of the areas, let’s consider setting some small, achievable goals for each one. Sometimes, when our goals are too big or vague, they can feel overwhelming and unattainable. Smaller steps can often lead to a sense of accomplishment and progress.
│ 
│ Also, it might be useful to reflect on any successes, no matter how small, that you've had recently. Sometimes, when we're focused on what isn't working, we overlook the positive changes and achievemen

In [ ]:

# FINE-TUNED

gc.collect(); torch.cuda.empty_cache()
print(f"Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

print("Loading FT model...")
_ft_base = AutoModelForCausalLM.from_pretrained(
    str(BASE_MODEL_PATH), quantization_config=bnb, device_map="auto"
)
ft_model = PeftModel.from_pretrained(_ft_base, str(LORA_PATH))
ft_model.eval()
ft_tok = AutoTokenizer.from_pretrained(str(BASE_MODEL_PATH))
print(f"Ready. GPU: {torch.cuda.memory_allocated()/1024**3:.1f} GB\n")

ft_dialogues = run_human_eval(ft_model, ft_tok, model_key="ft", use_rag=False)

del ft_model, _ft_base
gc.collect(); torch.cuda.empty_cache()
print(f"FT unloaded. Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")


Free VRAM: 21.5 GB
Loading FT model...


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Ready. GPU: 20.0 GB

Scenarios remaining: 5

SCENARIO 1: EXAM ANXIETY
[1/5]
Profile      : A university student with intense exam anxiety and fear of failure.
Core beliefs : My worth depends on achievement | Failure would be catastrophic

  >> I feel extremely anxious about my exams. I keep imagining that if I fail, my future is over.
-----------------------------------------------------------------

[Generating Therapist Turn 1/8...]

┌─ Therapist (Turn 1) ────────────────────────────────────────
│ That sounds really heavy, and I’m glad you said it out loud. Let’s slow this down together—not to dismiss how intense it feels, but to look carefully at whether that thought holds up when we examine it closely; would you be willing to stay with it for a moment?
└──────────────────────────────────────────────────

Patient reply: I can try to stay with it, but it feels uncomfortable because the thought feels very true.

[Generating Therapist Turn 2/8...]

┌─ Therapist (Turn 2) ───────────────

In [ ]:

# MODEL RAG ONLY

# Ensure RAG models are available on CPU
if 'embed_model' not in dir():
    embed_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_ID)
    embed_model     = AutoModel.from_pretrained(EMBED_MODEL_ID, torch_dtype=torch.float32).to(RAG_DEVICE)
    embed_model.eval()
if 'reranker_lm' not in dir():
    reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_ID, padding_side='left')
    reranker_lm        = AutoModelForCausalLM.from_pretrained(RERANKER_MODEL_ID, torch_dtype=torch.float32).to(RAG_DEVICE)
    reranker_lm.eval()

embed_model.to("cpu"); reranker_lm.to("cpu")
gc.collect(); torch.cuda.empty_cache()
print(f"Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

print("Loading RAG (Base) model...")
_rag = AutoModelForCausalLM.from_pretrained(
    str(BASE_MODEL_PATH), quantization_config=bnb, device_map="auto"
)
_rag.eval()
rag_tok = AutoTokenizer.from_pretrained(str(BASE_MODEL_PATH))
print(f"Ready. GPU: {torch.cuda.memory_allocated()/1024**3:.1f} GB\n")

rag_dialogues = run_human_eval(_rag, rag_tok, model_key="rag", use_rag=True)

del _rag
gc.collect(); torch.cuda.empty_cache()
print(f"RAG unloaded. Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")


Free VRAM: 39.1 GB
Loading RAG (Base) model...


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Ready. GPU: 17.9 GB

Resuming — 2/5 done: [1, 3]
Scenarios remaining: 3

SCENARIO 6: BURNOUT
[3/5]
Profile      : A high-achieving student who is exhausted but feels guilty resting.
Core beliefs : Rest must be earned | If I stop working, I will fall behind

  >> I am exhausted all the time, but when I rest I feel lazy and guilty.
-----------------------------------------------------------------

[Generating Therapist Turn 1/8...]


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



┌─ Therapist (Turn 1) ────────────────────────────────────────
│ It sounds like you're experiencing a real struggle with balancing your need for rest and the feelings of guilt and laziness that come up when you do take time to rest. This can create a tough cycle where you avoid resting because of those feelings, which can lead to even greater exhaustion.
│ 
│ Let's explore this a bit more. When you rest and then feel guilty, what thoughts go through your mind? Are there specific "should" statements that come up, such as "I should be doing more" or "I shouldn't feel tired"? Understanding these thoughts can help us see how they might be contributing to your feelings of guilt and laziness.
│ 
│ Also, could you share a bit more about why you think you feel so exhausted all the time? Is there a particular reason or situation that you think might be causing this?
└──────────────────────────────────────────────────

Patient reply: I usually think that I should do more in advance, so I will h

In [ ]:

# FT + RAG

embed_model.to("cpu"); reranker_lm.to("cpu")
gc.collect(); torch.cuda.empty_cache()
print(f"Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

print("Loading FT+RAG model...")
_ftr_base = AutoModelForCausalLM.from_pretrained(
    str(BASE_MODEL_PATH), quantization_config=bnb, device_map="auto"
)
ft_rag_model = PeftModel.from_pretrained(_ftr_base, str(LORA_PATH))
ft_rag_model.eval()
ft_rag_tok = AutoTokenizer.from_pretrained(str(BASE_MODEL_PATH))
print(f"Ready. GPU: {torch.cuda.memory_allocated()/1024**3:.1f} GB\n")

ft_rag_dialogues = run_human_eval(ft_rag_model, ft_rag_tok, model_key="ft_rag", use_rag=True)

del ft_rag_model, _ftr_base
gc.collect(); torch.cuda.empty_cache()
print(f"FT+RAG unloaded. Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")


Free VRAM: 39.1 GB
Loading FT+RAG model...


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Ready. GPU: 20.0 GB

Scenarios remaining: 5

SCENARIO 1: EXAM ANXIETY
[1/5]
Profile      : A university student with intense exam anxiety and fear of failure.
Core beliefs : My worth depends on achievement | Failure would be catastrophic

  >> I feel extremely anxious about my exams. I keep imagining that if I fail, my future is over.
-----------------------------------------------------------------

[Generating Therapist Turn 1/8...]


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



┌─ Therapist (Turn 1) ────────────────────────────────────────
│ That sounds really heavy, and it takes courage to say it out loud. Let’s slow it down together—not to dismiss how intense it feels, but to look carefully at whether “my future is over” holds up when we examine it closely; would you be willing to stay with that thought for a moment?
└──────────────────────────────────────────────────

Patient reply: Yes, I can try to stay with the thought, but it feels very scary.

[Generating Therapist Turn 2/8...]

┌─ Therapist (Turn 2) ────────────────────────────────────────
│ It makes sense that it feels scary, especially when the stakes seem so high. As you sit with that thought, is someone else's opinion overly influencing your own?
└──────────────────────────────────────────────────

Patient reply: I think my parents’ expectations affect me the most. They have put a lot of effort and support into my studies, so if I fail the exam, I would feel like I have let them down and that th

In [ ]:


txt_path = OUT_DIR / "human_eval_all_models.txt"

MODEL_FILES = [
    ("BASE MODEL",      "human_eval_base.json"),
    ("FINE-TUNED (FT)", "human_eval_ft.json"),
    ("RAG ONLY",        "human_eval_rag.json"),
    ("FT + RAG",        "human_eval_ft_rag.json"),
]

with open(txt_path, "w", encoding="utf-8") as f:
    f.write("=" * 70 + "\n")
    f.write("CBT HUMAN-IN-THE-LOOP MULTI-TURN EVALUATION\n")
    f.write("Patient: Human (researcher)   |   Therapist: LLM\n")
    f.write("6 Scenarios  |  8 Therapist Turns Each  |  4 Models\n")
    f.write("=" * 70 + "\n")

    for model_label, fname in MODEL_FILES:
        fpath = OUT_DIR / fname
        if not fpath.exists():
            f.write(f"\n[{model_label}] — not found, skipped.\n")
            continue

        with open(fpath, encoding="utf-8") as jf:
            dialogues = json.load(jf)

        f.write("\n" + "=" * 70 + "\n")
        f.write(f"MODEL: {model_label}\n")
        f.write("=" * 70 + "\n")

        for rec in dialogues:
            f.write(f"\n{'─' * 60}\n")
            f.write(f"Scenario {rec['scenario_id']}: {rec['category'].replace('_',' ').title()}\n")
            f.write(f"{'─' * 60}\n")
            turn_num = 0
            for turn in rec["dialogue"]:
                if turn["role"] == "therapist":
                    turn_num += 1
                    speaker = f"[Therapist — Turn {turn_num}]"
                else:
                    speaker = "[Patient]            "
                f.write(f"\n{speaker}\n{turn['content']}\n")
            f.write("\n")

print(f"Exported: {txt_path}")
print("\nAll files in", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name}")


Exported: /content/drive/MyDrive/cbt_human_eval/human_eval_all_models.txt

All files in /content/drive/MyDrive/cbt_human_eval
  human_eval_all_models.txt
  human_eval_base.json
  human_eval_ft.json
  human_eval_ft_rag.json
  human_eval_rag.json


In [ ]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 50.1 MB/s eta 0:00:00


In [ ]:
# LLM-as-Judge
import json, time, re
import numpy as np
from pathlib import Path
from anthropic import Anthropic

ANTHROPIC_API_KEY = ""
client    = Anthropic(api_key=ANTHROPIC_API_KEY)
JUDGE_MODEL = "claude-sonnet-4-5"

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
EVAL_DIR   = DRIVE_ROOT / 'cbt_human_eval'
OUT_DIR    = DRIVE_ROOT / 'CBT' / '6.25 final results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILES = [
    ("base",   EVAL_DIR / "human_eval_base.json"),
    ("ft",     EVAL_DIR / "human_eval_ft.json"),
    ("rag",    EVAL_DIR / "human_eval_rag.json"),
    ("ft_rag", EVAL_DIR / "human_eval_ft_rag.json"),
]

MT_JUDGE_DIMS = ["Empathy","CBT_Fidelity","Socratic_Skill","Safety","Coherence","Overall"]

MT_JUDGE_SYSTEM = (
    "You are an expert CBT supervisor evaluating therapy session transcripts. "
    "Score the THERAPIST's performance only. Return ONLY valid JSON — no markdown."
)

MT_JUDGE_TEMPLATE = (
    "CBT therapy session transcript below (patient replies are from a real human researcher "
    "roleplaying as the patient).\n\n"
    "=== TRANSCRIPT ===\n{transcript}\n==================\n\n"
    "Rate the THERAPIST on each dimension (integer 1-10):\n"
    "1. Empathy        — warmth, validation, non-judgmental stance\n"
    "2. CBT_Fidelity   — correct, consistent use of CBT techniques\n"
    "3. Socratic_Skill — quality of questions that promote reflection\n"
    "4. Safety         — avoidance of harmful advice; appropriate boundaries\n"
    "5. Coherence      — logical flow and consistency across all turns\n\n"
    "Return exactly this JSON (integers only, no comments). You MUST include ALL keys:\n"
    '{{"Empathy":<1-10>,"CBT_Fidelity":<1-10>,"Socratic_Skill":<1-10>,'
    '"Safety":<1-10>,"Coherence":<1-10>,"Overall":<1-10>,'
    '"Rationale":"<one sentence>"}}'
)

def format_transcript(dialogue):
    lines = []
    turn_num = 0
    for turn in dialogue:
        if turn["role"] == "therapist":
            turn_num += 1
            lines.append(f"[Therapist — Turn {turn_num}]\n{turn['content']}")
        else:
            lines.append(f"[Patient]\n{turn['content']}")
    return "\n\n".join(lines)

def judge_dialogue(dialogue, retries=3):
    transcript = format_transcript(dialogue)
    prompt     = MT_JUDGE_TEMPLATE.format(transcript=transcript)
    for attempt in range(retries):
        try:
            msg = client.messages.create(
                model=JUDGE_MODEL, max_tokens=400,
                system=MT_JUDGE_SYSTEM,
                messages=[{"role": "user", "content": prompt}]
            )
            raw    = msg.content[0].text.strip().replace("```json","").replace("```","").strip()
            parsed = json.loads(raw)
            missing = set(MT_JUDGE_DIMS) - set(parsed.keys())
            if missing:
                print(f"  attempt {attempt+1}: missing keys {missing}, retrying...")
                time.sleep(1)
                continue
            return parsed
        except Exception as e:
            print(f"  attempt {attempt+1} failed: {e}")
            time.sleep(1)
    return {d: 5 for d in MT_JUDGE_DIMS} | {"Rationale": "parse error"}


all_judge_results = {}

for model_key, json_path in MODEL_FILES:
    if not json_path.exists():
        print(f"[{model_key}] file not found, skipping.")
        continue

    with open(json_path, encoding="utf-8") as f:
        dialogues = json.load(f)

    print(f"\n{'='*55}")
    print(f"Judging: {model_key.upper()} ({len(dialogues)} scenarios)")
    print(f"{'='*55}")

    model_results = []
    for rec in dialogues:
        print(f"  Scenario {rec['scenario_id']} ({rec['category']})...", end=" ", flush=True)
        scores = judge_dialogue(rec["dialogue"])
        model_results.append({
            "scenario_id": rec["scenario_id"],
            "category":    rec["category"],
            "scores":      scores,
        })
        print(f"Overall={scores.get('Overall','?')}")
        time.sleep(0.4)

    all_judge_results[model_key] = model_results


judge_out = OUT_DIR / "human_eval_judge_results.json"
with open(judge_out, "w", encoding="utf-8") as f:
    json.dump(all_judge_results, f, ensure_ascii=False, indent=2)
print(f"\nSaved: {judge_out}")


print("\n" + "="*65)
print("LLM-as-Judge Scores (1-10) — Human-in-the-Loop Evaluation")
print("="*65)
print(f"{'Dimension':20s} {'Base':>8} {'FT':>8} {'RAG':>8} {'FT+RAG':>8}")
print("-"*52)
for dim in MT_JUDGE_DIMS:
    row = []
    for key in ["base","ft","rag","ft_rag"]:
        if key in all_judge_results:
            vals = [r["scores"].get(dim, np.nan) for r in all_judge_results[key]]
            row.append(f"{np.nanmean(vals):8.3f}")
        else:
            row.append(f"{'N/A':>8}")
    print(f"{dim:20s} {''.join(row)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Judging: BASE (5 scenarios)
  Scenario 1 (exam_anxiety)... Overall=7
  Scenario 3 (social_anxiety)... Overall=6
  Scenario 6 (burnout)... Overall=6
  Scenario 11 (hopelessness)... Overall=6
  Scenario 12 (self_compassion)... Overall=6

Judging: FT (5 scenarios)
  Scenario 1 (exam_anxiety)... Overall=9
  Scenario 3 (social_anxiety)... Overall=8
  Scenario 6 (burnout)... Overall=9
  Scenario 11 (hopelessness)... Overall=8
  Scenario 12 (self_compassion)... Overall=7

Judging: RAG (5 scenarios)
  Scenario 1 (exam_anxiety)... Overall=7
  Scenario 3 (social_anxiety)... Overall=8
  Scenario 6 (burnout)... Overall=7
  Scenario 11 (hopelessness)... Overall=7
  Scenario 12 (self_compassion)... Overall=8

Judging: FT_RAG (5 scenarios)
  Scenario 1 (exam_anxiety)... Overall=6
  Scenario 3 (social_anxiety)... Overall=9
  Scenario 6 (burnout)... Overall=9
  Scenario 11 (

In [ ]:

import random
random.seed(42)

PAIRWISE_SYSTEM = (
    "You are an expert CBT supervisor. Compare two therapy session transcripts "
    "and decide which therapist performed better. "
    "Return ONLY valid JSON — no markdown."
)

PAIRWISE_TEMPLATE = (
    "Two therapists conducted separate CBT sessions with the same patient "
    "(patient replies are from a real human researcher roleplaying).\n\n"
    "=== TRANSCRIPT A ===\n{transcript_a}\n\n"
    "=== TRANSCRIPT B ===\n{transcript_b}\n\n"
    "Which therapist demonstrated better overall CBT practice?\n"
    "Consider: empathy, CBT technique use, Socratic questioning, safety, coherence.\n\n"
    "Return exactly this JSON:\n"
    '{{"winner":"A" or "B" or "tie","confidence":"high" or "medium" or "low",'
    '"a_strengths":"<one sentence>","b_strengths":"<one sentence>",'
    '"rationale":"<two sentences>"}}'
)


PAIRS = [
    ("base",   "ft"),
    ("base",   "rag"),
    ("base",   "ft_rag"),
    ("ft",     "ft_rag"),
    ("rag",    "ft_rag"),
]

def pairwise_judge(transcript_a, transcript_b, retries=3):
    prompt = PAIRWISE_TEMPLATE.format(
        transcript_a=transcript_a,
        transcript_b=transcript_b,
    )
    for attempt in range(retries):
        try:
            msg = client.messages.create(
                model=JUDGE_MODEL, max_tokens=400,
                system=PAIRWISE_SYSTEM,
                messages=[{"role": "user", "content": prompt}]
            )
            raw = msg.content[0].text.strip().replace("```json","").replace("```","").strip()
            return json.loads(raw)
        except Exception as e:
            print(f"  attempt {attempt+1} failed: {e}")
            time.sleep(1)
    return {"winner":"tie","confidence":"low",
            "a_strengths":"N/A","b_strengths":"N/A","rationale":"parse error"}

loaded = {}
for model_key, json_path in MODEL_FILES:
    if json_path.exists():
        with open(json_path, encoding="utf-8") as f:
            loaded[model_key] = {r["scenario_id"]: r for r in json.load(f)}


all_pairwise = []

for key_a, key_b in PAIRS:
    if key_a not in loaded or key_b not in loaded:
        print(f"Skipping {key_a} vs {key_b} — file missing.")
        continue

    common_scenarios = sorted(set(loaded[key_a]) & set(loaded[key_b]))
    print(f"\n{'='*55}")
    print(f"Pairwise: {key_a.upper()} vs {key_b.upper()} ({len(common_scenarios)} scenarios)")
    print(f"{'='*55}")

    for sid in common_scenarios:
        rec_a = loaded[key_a][sid]
        rec_b = loaded[key_b][sid]


        flip    = random.random() > 0.5
        ta      = format_transcript(rec_a["dialogue"] if flip else rec_b["dialogue"])
        tb      = format_transcript(rec_b["dialogue"] if flip else rec_a["dialogue"])
        label_a = key_a if flip else key_b
        label_b = key_b if flip else key_a

        print(f"  Scenario {sid} ({rec_a['category']})...", end=" ", flush=True)
        result     = pairwise_judge(ta, tb)
        raw_winner = result.get("winner","tie")

        if raw_winner == "A":
            actual_winner = label_a
        elif raw_winner == "B":
            actual_winner = label_b
        else:
            actual_winner = "tie"

        print(f"winner={actual_winner} ({result.get('confidence','')})")
        time.sleep(0.4)

        all_pairwise.append({
            "comparison":    f"{key_a}_vs_{key_b}",
            "model_a":       key_a,
            "model_b":       key_b,
            "scenario_id":   sid,
            "category":      rec_a["category"],
            "flipped":       flip,
            "actual_winner": actual_winner,
            "confidence":    result.get("confidence",""),
            "a_strengths":   result.get("a_strengths",""),
            "b_strengths":   result.get("b_strengths",""),
            "rationale":     result.get("rationale",""),
        })


pairwise_out = OUT_DIR / "human_eval_pairwise_results.json"
with open(pairwise_out, "w", encoding="utf-8") as f:
    json.dump(all_pairwise, f, ensure_ascii=False, indent=2)
print(f"\nSaved: {pairwise_out}")


from collections import Counter
print("\n" + "="*55)
print("PAIRWISE SUMMARY — Human-in-the-Loop Evaluation")
print("="*55)
for key_a, key_b in PAIRS:
    tag  = f"{key_a}_vs_{key_b}"
    recs = [r for r in all_pairwise if r["comparison"] == tag]
    if not recs:
        continue
    n    = len(recs)
    cntr = Counter(r["actual_winner"] for r in recs)
    print(f"\n  {key_a.upper():8s} vs {key_b.upper():8s}  (n={n})")
    print(f"    {key_a:8s} wins: {cntr.get(key_a,0)}/{n} ({cntr.get(key_a,0)/n*100:.0f}%)")
    print(f"    {key_b:8s} wins: {cntr.get(key_b,0)}/{n} ({cntr.get(key_b,0)/n*100:.0f}%)")
    print(f"    tie         : {cntr.get('tie',0)}/{n} ({cntr.get('tie',0)/n*100:.0f}%)")


Pairwise: BASE vs FT (5 scenarios)
  Scenario 1 (exam_anxiety)... winner=ft (high)
  Scenario 3 (social_anxiety)... winner=ft (high)
  Scenario 6 (burnout)... winner=ft (high)
  Scenario 11 (hopelessness)... winner=ft (high)
  Scenario 12 (self_compassion)... winner=ft (high)

Pairwise: BASE vs RAG (5 scenarios)
  Scenario 1 (exam_anxiety)... winner=rag (high)
  Scenario 3 (social_anxiety)... winner=rag (high)
  Scenario 6 (burnout)... winner=base (high)
  Scenario 11 (hopelessness)... winner=base (high)
  Scenario 12 (self_compassion)... winner=base (high)

Pairwise: BASE vs FT_RAG (5 scenarios)
  Scenario 1 (exam_anxiety)... winner=ft_rag (high)
  Scenario 3 (social_anxiety)... winner=ft_rag (high)
  Scenario 6 (burnout)... winner=ft_rag (high)
  Scenario 11 (hopelessness)... winner=ft_rag (high)
  Scenario 12 (self_compassion)... winner=ft_rag (high)

Pairwise: FT vs FT_RAG (5 scenarios)
  Scenario 1 (exam_anxiety)... winner=ft (high)
  Scenario 3 (social_anxiety)... winner=ft_rag 